# Layout-Aware Chunking Demo

This notebook demonstrates how the `scholar_rag` layout-aware chunking pipeline processes a PDF document.

We will load a mock arXiv PDF, pass it through the `LayoutAwareChunker` (which uses `docling`), and observe how tables, headers, and paragraphs are preserved and sectioned.

In [1]:
import os
import sys
from pathlib import Path

# Add the project root to the path so we can import scholar_rag
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import logging
logging.basicConfig(level=logging.INFO)

## 1. Load the Mock Document
We use the `LocalDirectoryReader` to fetch a parsed Document from the `mock_data` directory.

In [13]:
from scholar_rag.ingestion.local import LocalDirectoryReader

mock_data_dir = os.path.join(project_root, "mock_data")
reader = LocalDirectoryReader(mock_data_dir)

# Read just one document
documents = list(reader.read())
if not documents:
    print("No documents found in mock_data!")
else:
    sample_doc = documents[1]
    print(f"Loaded Document ID: {sample_doc.id}")
    print(f"Title: {sample_doc.metadata.get('title')}")
    print(f"Source Path: {sample_doc.metadata.get('source_path')}")

Loaded Document ID: 7d7ca9ee72708f15b68462dcd6716e6d1021196b35c9644ada6658a419c4995c
Title: Atoms of Thought: Universal EEG Representation Learning with Microstates
Source Path: /Users/tri/Projects/scholar-rag/mock_data/2605.20182v1_Atoms_of_Thought_Universal_EEG_Representation_Lear.pdf


## 2. Process with Layout-Aware Chunker
Now we pass the document to the chunker. The chunker will parse the original PDF file structurally.

In [14]:
from scholar_rag.chunking import LayoutAwareChunker

chunker = LayoutAwareChunker(max_chunk_size=1500)
chunks = list(chunker.chunk(sample_doc))

print(f"Successfully split document into {len(chunks)} chunks.")

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 79f7a7da676ef83f364170dd36130ce6
INFO:docling.models.stages.ocr.auto_ocr_model:ocrmac cannot be used because ocrmac is not installed.
INFO:docling.models.stages.ocr.auto_ocr_model:rapidocr cannot be used because onnxruntime is not installed.
INFO:docling.models.stages.ocr.auto_ocr_model:easyocr cannot be used because it is not installed.
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
[INFO] 2026-05-21 17:29:11,333 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-21 17:29:11,334 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-21 17:29:11,345 [RapidOCR] download_file.py:60: File exists and is valid: /Users/tri/Projects/scholar-rag/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document 2605.20182v1_Atoms_of_Thought_Universal_EEG_Representation_Lear.pdf
INFO:docling.document_converter:Finished converting document 2605.20182v1_Atoms_of_Thought_Universal_EEG_Representation_Lear.pdf in 69.03 sec.


Successfully split document into 306 chunks.


## 3. Visualize the Chunks
Let's look at the first few chunks to see how the layout was preserved.

In [16]:
from IPython.display import Markdown, display

# Display first 5 chunks
for i, chunk in enumerate(chunks[0:200]):
    strategy = chunk.metadata.get('chunking_strategy')
    element_type = chunk.metadata.get('element_type')
    
    output = f"### Chunk {i} (Strategy: {strategy}, Type: {element_type})\n"
    output += f"**ID:** `{chunk.id}`\n\n"
    output += f"```text\n{chunk.content}\n```\n"
    output += "---"
    
    display(Markdown(output))

### Chunk 0 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `78c6bfbf806a85cbf2001b8da222e6b9a9fc8b124084c40821a313999f418f51`

```text
[Xinyang Tian ∗](https://orcid.org/0009-0008-3168-523X)
```
---

### Chunk 1 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `906c96063a1125e18bcc476a28da47f475bad8b4abc85c42340487d569d75ca8`

```text
Institute for Interdisciplinary Information Sciences, Tsinghua University Beijing, China xinyangtian368@gmail.com
```
---

### Chunk 2 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `c692acbee7c3480559baf3a4e4ef3f6706fab20678ef24bbb85f96b8d6f0ecba`

```text
Siyang Xue
```
---

### Chunk 3 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `e3535bf6b8cb20e718e9770ff405938dc2398ff57221923b5ba0da84a720c8f6`

```text
School of Clinical Medicine, Tsinghua University Beijing, China xuesy22@mails.tsinghua.edu.cn
```
---

### Chunk 4 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `034501b5e8c9c6fdc2b4fde4cbdf4b6531c8cb17f30d73c0e60a427c862192fe`

```text
Learning universal representations from electroencephalogram (EEG) signals is a cutting-edge approach in the field of neuroinformatics and brain-computer interfaces (BCIs). Conventionally, EEG is treated as a multivariate temporal signal, where time- or frequency-domain features are extracted for representation learning. This paper investigates a simple yet effective EEG representation, i.e., microstates. Microstates represent the building blocks of brain activity patterns at a microscopic time scale. We build a universal microstate tokenizer from a large medical EEG dataset by clustering continuous EEG signals into sequences of discrete microstates. The microstate tokenizer is then adopted universally across a series of downstream tasks, including sleep staging, emotion recognition, and motor imagery classification. Experimental results show that EEG representation learning with microstates outperforms traditional time-domain and frequency-domain features under different models and across different tasks. Further analysis shows that microstates offer greater interpretability and scalability, thereby opening up applications in both cognitive neuroscience and clinical research.
```
---

...

### Chunk 195 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `f55221dda93ddf387e04444ffab9109dc98465c29162709e0011e69c2aa257df`

```text
<!-- formula-not-decoded -->
```
---

### Chunk 196 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `204929c64c6b59186487f2b83b3e296a66ac9e750fd6d100a538a27814737944`

```text
and since this value is calculated in parallel, we have to incorporate the positional information 𝑖 along with 𝒙 𝑖 into the queries and keys.
```
---

### Chunk 197 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `b4d7cbc4d5c9986d71567f38bc0bf3259c18c6f9c6d974491cddc3702e5e408a`

```text
The main idea of RoFormer is to select a positional embedding such that
```
---

### Chunk 198 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `f55221dda93ddf387e04444ffab9109dc98465c29162709e0011e69c2aa257df`

```text
<!-- formula-not-decoded -->
```
---

### Chunk 199 (Strategy: layout_aware, Type: LayoutBlock)
**ID:** `74ff6caebe47285d5bdee4635a03c989b0a9d38a4d9ba6018054e6ea9a2998c4`

```text
is a function that depends solely on the input word vector and the relative position between 𝑚,𝑛 .
```
---